# LTA × Nebula Hackathon — ACV Final Anti-Overfitting Pipeline

This is the stricter final ACV notebook.

## What changed from the earlier version

The pipeline now enforces a clear **train → validate → freeze → test** boundary:

- feature choices are fixed using training data and ACV domain logic,
- supervised models are trained only on labelled training cases,
- validation is by **whole case**, never by timestamp row,
- Case 06 is explicitly included,
- Case 04 is kept out of the main standard-schema model because its telemetry schema is materially different,
- ensemble weights are **equal and fixed**, rather than tuned to the hidden test file,
- no test file is read until the configuration is frozen,
- 30-minute and 60-minute persistence analysis is included,
- feature ablation and optional stress testing are included,
- the final trained model bundle and competition CSV are exported.

The purpose is not to chase a prettier training score. The purpose is to make the final ranking as defensible and robust as possible.

## 1. Setup

In [ ]:
# If required:
# %pip install pandas numpy openpyxl matplotlib scikit-learn joblib

from pathlib import Path
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
print("Setup complete.")

## 2. Paths

The notebook uses your local ACV folder if it exists. It also supports a flat `/mnt/data` layout for portability/testing.

In [ ]:
LOCAL_ROOT = Path("/Users/ashwin/Desktop/ACV")
FALLBACK_ROOT = Path("/mnt/data")

if (LOCAL_ROOT / "Train").exists():
    DATA_ROOT = LOCAL_ROOT
    TRAIN_DIR = DATA_ROOT / "Train"
    TEST_DIR = DATA_ROOT / "Test"
    FLAT_LAYOUT = False
else:
    DATA_ROOT = FALLBACK_ROOT
    TRAIN_DIR = DATA_ROOT
    TEST_DIR = DATA_ROOT
    FLAT_LAYOUT = True

OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def train_path(filename):
    return TRAIN_DIR / filename

def test_path(filename="acv_test_case.xlsx"):
    p = TEST_DIR / filename
    if p.exists():
        return p

    # Handle the duplicate filename used in the uploaded working copy.
    alt = TEST_DIR / "acv_test_case(1).xlsx"
    if alt.exists():
        return alt

    raise FileNotFoundError("Could not locate the ACV test file.")

print("DATA_ROOT:", DATA_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 3. Load training labels only

The hidden test label is never available and never used.

A fallback label table is included only in case `Train_Labels.csv` is not beside the dataset.

In [ ]:
labels_file = DATA_ROOT / "Train_Labels.csv"

if labels_file.exists():
    labels = pd.read_csv(labels_file)
else:
    labels = pd.DataFrame({
        "filename": [
            "acv_case_01.xlsx",
            "acv_case_02.xlsx",
            "acv_case_03.xlsx",
            "acv_case_04.xlsx",
            "acv_case_05.xlsx",
            "acv_case_06.xlsx",
        ],
        "faulty_car": ["01", "02", "03", "01", "04", "06"],
    })

def normalise_car_id(value):
    match = re.search(r"(\d+)", str(value))
    if not match:
        raise ValueError(f"Could not parse car ID from {value!r}")
    return f"{int(match.group(1)):02d}"

labels["faulty_car"] = labels["faulty_car"].map(normalise_car_id)
labels

## 4. Training-only schema audit

We determine the usable standard-schema cases from the **training files only**.

Case 04 has a rich 63-parameter-per-car schema, so it is excluded from the main standard-schema model rather than forcing uncertain sensor mappings.

In [ ]:
def extract_car_ids(columns):
    cars = set()

    for col in columns:
        m = re.match(r"Car (\d{2}) - ", str(col))
        if m:
            cars.add(m.group(1))

    return sorted(cars)


def extract_parameter_names(columns):
    params = set()

    for col in columns:
        m = re.match(r"Car \d{2} - (.+)", str(col))
        if m:
            params.add(m.group(1))

    return params


training_files = sorted(DATA_ROOT.glob("acv_case_*.xlsx")) if FLAT_LAYOUT else sorted(TRAIN_DIR.glob("acv_case_*.xlsx"))

schema_rows = []
parameters_by_file = {}

for path in training_files:
    header = pd.read_excel(path, nrows=0)

    params = extract_parameter_names(header.columns)
    parameters_by_file[path.name] = params

    schema_rows.append({
        "filename": path.name,
        "cars": len(extract_car_ids(header.columns)),
        "parameters_per_car": len(params),
        "total_columns": len(header.columns),
    })

schema_audit = pd.DataFrame(schema_rows)
schema_audit

In [ ]:
CORE_STANDARD_PARAMETERS = {
    "ACV Control Temperature (Cooling)",
    "ACV Control Temperature (Heating)",
    "ACV Information Valid",
    "ACV Running Mode",
    "ACV Setting Mode",
    "Indoor Average Temperature",
    "Load Halved",
}

STANDARD_CASES = sorted([
    filename
    for filename, params in parameters_by_file.items()
    if CORE_STANDARD_PARAMETERS.issubset(params)
])

print("Standard-schema training cases:")
for f in STANDARD_CASES:
    print("-", f)

assert "acv_case_06.xlsx" in STANDARD_CASES, "Case 06 must be included."
assert "acv_case_04.xlsx" not in STANDARD_CASES, "Case 04 should remain outside the standard-schema model."
assert len(STANDARD_CASES) == 5, f"Expected 5 standard cases, got {len(STANDARD_CASES)}."

## 5. Feature engineering

The model compares every car with its peers in the same train.

Key ideas:

- `cooling_gap = indoor temperature - cooling control temperature`
- compare each car against the **same-timestamp median of the other cars**
- examine whether the car remains abnormal while ACV is actively cooling
- use persistence fractions, not just one extreme point
- filter telemetry using `ACV Information Valid`

Outside temperature is supported under both names found in the training set:
`Outdoor Average Temperature` and `Outside Temperature Sensor Reading`.

In [ ]:
def find_column(df, car_id, possible_names):
    prefix = f"Car {car_id} - "

    for name in possible_names:
        col = prefix + name

        if col in df.columns:
            return col

    return None


def numeric_series(df, column, valid_mask=None):
    if column is None:
        return pd.Series(np.nan, index=df.index, dtype=float)

    s = pd.to_numeric(df[column], errors="coerce")

    if valid_mask is not None:
        s = s.where(valid_mask)

    return s


def extract_case_features(df):
    cars = extract_car_ids(df.columns)

    indoor = {}
    cooling_target = {}
    outdoor = {}
    running = {}
    valid = {}
    cooling_gap = {}

    for car in cars:
        valid_col = find_column(df, car, ["ACV Information Valid"])
        valid_mask = df[valid_col].eq("Valid") if valid_col else pd.Series(True, index=df.index)

        valid[car] = valid_mask

        indoor[car] = numeric_series(
            df,
            find_column(df, car, ["Indoor Average Temperature"]),
            valid_mask,
        )

        cooling_target[car] = numeric_series(
            df,
            find_column(df, car, ["ACV Control Temperature (Cooling)"]),
            valid_mask,
        )

        outdoor[car] = numeric_series(
            df,
            find_column(
                df,
                car,
                [
                    "Outdoor Average Temperature",
                    "Outside Temperature Sensor Reading",
                ],
            ),
            valid_mask,
        )

        running_col = find_column(df, car, ["ACV Running Mode"])

        if running_col:
            running[car] = df[running_col].where(valid_mask)
        else:
            running[car] = pd.Series(np.nan, index=df.index, dtype=object)

        cooling_gap[car] = indoor[car] - cooling_target[car]

    indoor_df = pd.DataFrame(indoor)
    gap_df = pd.DataFrame(cooling_gap)

    # Median across the train at the same timestamp.
    peer_indoor_median = indoor_df.median(axis=1, skipna=True)
    peer_gap_median = gap_df.median(axis=1, skipna=True)

    rows = []

    for car in cars:
        ind = indoor[car]
        target = cooling_target[car]
        out = outdoor[car]
        gap = cooling_gap[car]
        run = running[car]

        active = run.isin(["Automatic Cooling", "Full Cooling"])
        active_gap = gap.where(active)

        rel_indoor = ind - peer_indoor_median
        rel_gap = gap - peer_gap_median

        rows.append({
            "car": car,

            "valid_fraction": valid[car].mean(),

            "indoor_mean": ind.mean(),
            "indoor_median": ind.median(),
            "indoor_std": ind.std(),
            "indoor_p90": ind.quantile(0.90),

            "cooling_target_mean": target.mean(),

            "cooling_gap_mean": gap.mean(),
            "cooling_gap_median": gap.median(),
            "cooling_gap_std": gap.std(),
            "cooling_gap_p90": gap.quantile(0.90),
            "cooling_gap_p95": gap.quantile(0.95),

            "active_gap_mean": active_gap.mean(),
            "active_gap_p90": active_gap.quantile(0.90),

            "outdoor_indoor_gap_mean": (out - ind).mean(),

            "rel_indoor_mean": rel_indoor.mean(),
            "rel_indoor_p90": rel_indoor.quantile(0.90),
            "rel_gap_mean": rel_gap.mean(),
            "rel_gap_p90": rel_gap.quantile(0.90),

            # Persistence: how often this car is above its peers.
            "rel_indoor_pos_frac": (rel_indoor > 0).where(rel_indoor.notna()).mean(),
            "rel_gap_pos_frac": (rel_gap > 0).where(rel_gap.notna()).mean(),

            "automatic_cooling_fraction": run.eq("Automatic Cooling").mean(),
            "full_cooling_fraction": run.eq("Full Cooling").mean(),
            "stop_fraction": run.eq("Stop").mean(),
        })

    return pd.DataFrame(rows)

## 6. Build the labelled training feature table

The independent unit is a whole fault case.  
The thousands of 30-second rows are **not** treated as thousands of independent fault examples.

In [ ]:
training_raw = {}
feature_tables = []

for filename in STANDARD_CASES:
    df = pd.read_excel(train_path(filename))
    training_raw[filename] = df

    faulty_car = labels.loc[
        labels["filename"].eq(filename),
        "faulty_car",
    ].iloc[0]

    features = extract_case_features(df)

    features["filename"] = filename
    features["faulty_car"] = faulty_car
    features["faulty"] = features["car"].eq(faulty_car).astype(int)

    feature_tables.append(features)

feature_df = pd.concat(feature_tables, ignore_index=True)

print("Feature rows:", len(feature_df))
print("Faulty examples:", feature_df["faulty"].sum())
print("Healthy examples:", (feature_df["faulty"] == 0).sum())

case_check = feature_df.groupby("filename")["faulty"].sum()
print("\nFaults per case:")
print(case_check)

assert (case_check == 1).all()
assert set(feature_df["filename"].unique()) == set(STANDARD_CASES)

feature_df.head()

## 7. Fixed robust feature set

These features are fixed before the test file is loaded.

They combine absolute thermal performance, peer-relative behaviour, and persistence.

In [ ]:
ROBUST_FEATURES = [
    "indoor_mean",
    "cooling_gap_mean",
    "cooling_gap_p90",
    "active_gap_mean",
    "rel_indoor_mean",
    "rel_gap_mean",
    "rel_indoor_pos_frac",
    "rel_gap_pos_frac",
]

MODEL_FEATURES = ROBUST_FEATURES.copy()

ROBUST_FEATURES

## 8. Domain score and official rank-decay metric

In [ ]:
def add_domain_score(case_features, selected_features=None):
    selected_features = selected_features or ROBUST_FEATURES

    scored = case_features.copy()
    percentile_cols = []

    for feature in selected_features:
        pct_col = f"{feature}_pct"

        if scored[feature].notna().any():
            scored[pct_col] = (
                scored[feature]
                .rank(pct=True, ascending=True, method="average")
                .fillna(0.5)
            )
        else:
            scored[pct_col] = 0.5

        percentile_cols.append(pct_col)

    scored["domain_score"] = scored[percentile_cols].mean(axis=1)

    return scored


def normalized_rank_score(values):
    values = pd.Series(values)

    ranks = values.rank(
        ascending=False,
        method="average",
    )

    n = len(values)

    if n <= 1:
        return pd.Series(np.ones(n), index=values.index)

    # 1.0 = first, 0.0 = last.
    return (n - ranks) / (n - 1)


def rank_decay_score(true_car, ranked_cars):
    if true_car not in ranked_cars:
        return 0.0

    n = len(ranked_cars)
    r = ranked_cars.index(true_car) + 1

    return (n - (r - 1)) / n

## 9. Fixed conservative supervised models

Hyperparameters are deliberately conservative because there are only five directly comparable independent fault cases.

No hyperparameter search is performed on the test file.

In [ ]:
MODELS = {
    "logistic": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            C=0.5,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),

    "rf": RandomForestClassifier(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "extra": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}


def prepare_matrices(train_x, test_x):
    train_x = train_x.copy()
    test_x = test_x.copy()

    medians = train_x.median(numeric_only=True)

    train_x = train_x.fillna(medians).fillna(0)
    test_x = test_x.fillna(medians).fillna(0)

    return train_x, test_x, medians

## 10. 30-minute and 60-minute persistence scoring

A true refrigeration problem should ideally remain suspicious across multiple windows instead of appearing only because of one brief spike.

Window scoring uses the same fixed domain features.

In [ ]:
def window_aggregate(df, minutes, min_rows=20):
    data = df.copy()
    data["Time"] = pd.to_datetime(data["Time"])

    window_rows = []

    for _, window in data.groupby(
        pd.Grouper(
            key="Time",
            freq=f"{minutes}min",
        )
    ):
        if len(window) < min_rows:
            continue

        features = extract_case_features(window)

        if features.empty:
            continue

        scored = add_domain_score(features)

        scored["top1"] = scored["domain_score"].eq(
            scored["domain_score"].max()
        ).astype(float)

        window_rows.append(
            scored[["car", "domain_score", "top1"]]
        )

    cars = extract_car_ids(df.columns)

    if not window_rows:
        return pd.DataFrame({
            "car": cars,
            "window_score": 0.5,
            "top1_freq": np.nan,
            "windows": 0,
        })

    all_windows = pd.concat(window_rows, ignore_index=True)

    result = (
        all_windows
        .groupby("car")
        .agg(
            mean_window_domain=("domain_score", "mean"),
            top1_freq=("top1", "mean"),
            windows=("top1", "size"),
        )
        .reset_index()
    )

    # Fixed combination, not tuned on the test case.
    result["window_score"] = (
        0.70 * result["mean_window_domain"]
        + 0.30 * result["top1_freq"]
    )

    return result

## 11. Lock the ensemble BEFORE touching the test file

To reduce model-selection overfitting, all six components receive **equal weight**.

This is intentionally simpler than searching for weights that maximise performance on only five cases.

In [ ]:
LOCKED_METHOD_WEIGHTS = {
    "domain": 1 / 6,
    "logistic": 1 / 6,
    "rf": 1 / 6,
    "extra": 1 / 6,
    "window30": 1 / 6,
    "window60": 1 / 6,
}

LOCKED_CONFIG = {
    "standard_cases": STANDARD_CASES,
    "robust_features": ROBUST_FEATURES,
    "model_features": MODEL_FEATURES,
    "ensemble_weights": LOCKED_METHOD_WEIGHTS,
    "random_state": RANDOM_STATE,
    "test_used_for_selection": False,
}

with open(OUTPUT_DIR / "acv_locked_config.json", "w") as f:
    json.dump(LOCKED_CONFIG, f, indent=2)

LOCKED_CONFIG

## 12. Whole-case leave-one-case-out validation of the LOCKED ensemble

For each round:

- one entire case is held out,
- supervised models train on the other four cases,
- the held-out case is ranked,
- the official ACV rank-decay score is calculated.

The held-out raw case is used only for inference/window scoring, never for fitting the supervised model.

In [ ]:
def ensemble_case(
    train_features,
    inference_features,
    inference_raw,
    method_weights=LOCKED_METHOD_WEIGHTS,
):
    result = inference_features[["car"]].copy()

    # 1. Domain ranking
    domain = add_domain_score(inference_features)
    result["domain"] = normalized_rank_score(
        domain["domain_score"]
    ).values

    # 2. Supervised models
    X_train, X_infer, _ = prepare_matrices(
        train_features[MODEL_FEATURES],
        inference_features[MODEL_FEATURES],
    )

    y_train = train_features["faulty"]

    for name, base_model in MODELS.items():
        model = clone(base_model)
        model.fit(X_train, y_train)

        probability = model.predict_proba(X_infer)[:, 1]

        result[name] = normalized_rank_score(
            probability
        ).values

    # 3. Window persistence
    for minutes, name in [
        (30, "window30"),
        (60, "window60"),
    ]:
        window_scores = window_aggregate(
            inference_raw,
            minutes=minutes,
        )

        merged = result[["car"]].merge(
            window_scores[["car", "window_score"]],
            on="car",
            how="left",
        )

        result[name] = normalized_rank_score(
            merged["window_score"].fillna(0.5)
        ).values

    # Equal-weight frozen consensus
    result["ensemble_score"] = sum(
        method_weights[name] * result[name]
        for name in method_weights
    )

    return (
        result
        .sort_values("ensemble_score", ascending=False)
        .reset_index(drop=True)
    )


validation_rows = []

for holdout_file in STANDARD_CASES:
    train_features = feature_df[
        ~feature_df["filename"].eq(holdout_file)
    ].copy()

    val_features = feature_df[
        feature_df["filename"].eq(holdout_file)
    ].copy()

    true_car = val_features["faulty_car"].iloc[0]

    ranking = ensemble_case(
        train_features=train_features,
        inference_features=val_features,
        inference_raw=training_raw[holdout_file],
    )

    ranked_cars = ranking["car"].tolist()
    true_rank = ranked_cars.index(true_car) + 1

    validation_rows.append({
        "filename": holdout_file,
        "true_faulty_car": true_car,
        "true_rank": true_rank,
        "rank_decay_score": rank_decay_score(
            true_car,
            ranked_cars,
        ),
        "ranked_cars": "|".join(ranked_cars),
    })

validation = pd.DataFrame(validation_rows)
validation

In [ ]:
print(
    "Mean locked-ensemble rank-decay:",
    validation["rank_decay_score"].mean()
)

print(
    "Top-1 cases:",
    (validation["true_rank"] == 1).sum(),
    "/",
    len(validation),
)

print(
    "Mean true rank:",
    validation["true_rank"].mean()
)

assert "acv_case_06.xlsx" in validation["filename"].values

## 13. Feature-ablation stability test

Remove one domain feature at a time and check whether the labelled fault still ranks first.

This tests whether the result depends on one lucky signal.

In [ ]:
ablation_rows = []

for removed_feature in ROBUST_FEATURES:
    selected = [
        f for f in ROBUST_FEATURES
        if f != removed_feature
    ]

    ranks = []

    for filename in STANDARD_CASES:
        case_features = feature_df[
            feature_df["filename"].eq(filename)
        ].copy()

        true_car = case_features["faulty_car"].iloc[0]

        scored = add_domain_score(
            case_features,
            selected_features=selected,
        )

        ranked = (
            scored
            .sort_values("domain_score", ascending=False)["car"]
            .tolist()
        )

        ranks.append(
            ranked.index(true_car) + 1
        )

    ablation_rows.append({
        "removed_feature": removed_feature,
        "top1_cases": sum(r == 1 for r in ranks),
        "mean_true_rank": np.mean(ranks),
        "ranks": ranks,
    })

ablation_results = pd.DataFrame(ablation_rows)
ablation_results

## 14. Optional raw-data stress testing

This section is deliberately training-only.

It perturbs the held-out training case by:

- masking complete ACV telemetry rows,
- adding Gaussian noise to temperature-related signals.

Set `RUN_STRESS_TESTS = True` if you want to rerun the robustness suite. It can take a few minutes.

In [ ]:
RUN_STRESS_TESTS = False
STRESS_REPEATS = 5

STRESS_CONDITIONS = [
    {"name": "10pct_missing", "missing_fraction": 0.10, "noise_sd": 0.0},
    {"name": "20pct_missing", "missing_fraction": 0.20, "noise_sd": 0.0},
    {"name": "30pct_missing", "missing_fraction": 0.30, "noise_sd": 0.0},
    {"name": "noise_0.5C", "missing_fraction": 0.0, "noise_sd": 0.5},
    {"name": "20pct_missing_plus_0.5C", "missing_fraction": 0.20, "noise_sd": 0.5},
]


def perturb_case(df, missing_fraction=0.0, noise_sd=0.0, rng=None):
    rng = rng or np.random.default_rng(RANDOM_STATE)

    out = df.copy()

    # Simulate telemetry outages by masking whole ACV rows while preserving identifiers/time.
    if missing_fraction > 0:
        n_rows = len(out)
        n_mask = int(round(n_rows * missing_fraction))

        mask_indices = rng.choice(
            out.index.to_numpy(),
            size=n_mask,
            replace=False,
        )

        out.loc[
            mask_indices,
            out.columns[3:],
        ] = np.nan

    # Perturb temperature-like numeric measurements.
    if noise_sd > 0:
        temperature_cols = [
            col for col in out.columns
            if "Temperature" in str(col)
        ]

        for col in temperature_cols:
            numeric = pd.to_numeric(
                out[col],
                errors="coerce",
            )

            noise = rng.normal(
                loc=0.0,
                scale=noise_sd,
                size=len(out),
            )

            out[col] = numeric + noise

    return out


stress_summary = None

if RUN_STRESS_TESTS:
    stress_rows = []

    for condition in STRESS_CONDITIONS:
        for repeat in range(STRESS_REPEATS):
            rng = np.random.default_rng(
                RANDOM_STATE + repeat
            )

            for holdout_file in STANDARD_CASES:
                train_features = feature_df[
                    ~feature_df["filename"].eq(holdout_file)
                ].copy()

                perturbed_raw = perturb_case(
                    training_raw[holdout_file],
                    missing_fraction=condition["missing_fraction"],
                    noise_sd=condition["noise_sd"],
                    rng=rng,
                )

                perturbed_features = extract_case_features(
                    perturbed_raw
                )

                true_car = labels.loc[
                    labels["filename"].eq(holdout_file),
                    "faulty_car",
                ].iloc[0]

                ranking = ensemble_case(
                    train_features=train_features,
                    inference_features=perturbed_features,
                    inference_raw=perturbed_raw,
                )

                ranked = ranking["car"].tolist()
                true_rank = ranked.index(true_car) + 1

                stress_rows.append({
                    "condition": condition["name"],
                    "repeat": repeat,
                    "filename": holdout_file,
                    "true_rank": true_rank,
                    "rank_decay_score": rank_decay_score(
                        true_car,
                        ranked,
                    ),
                })

    stress_results = pd.DataFrame(stress_rows)

    stress_summary = (
        stress_results
        .groupby("condition")
        .agg(
            mean_rank_decay=("rank_decay_score", "mean"),
            top1_rate=("true_rank", lambda s: (s == 1).mean()),
            mean_true_rank=("true_rank", "mean"),
        )
        .sort_values("mean_rank_decay", ascending=False)
    )

    display(stress_summary)

else:
    print(
        "Stress tests are built in but not run by default. "
        "Set RUN_STRESS_TESTS = True and rerun this cell."
    )

# TEST BOUNDARY

Everything above this point uses **training data only**.

The model design, feature set, hyperparameters, ensemble weights, validation strategy, ablation checks, and optional stress-test logic are now frozen.

Only from this point onward do we read the unlabelled test telemetry.

## 15. First access to the unlabelled test file

In [ ]:
TEST_FILE = test_path()

print("Test file:", TEST_FILE.name)

test_df = pd.read_excel(TEST_FILE)

test_features = extract_case_features(test_df)

print("Detected cars:", test_features["car"].tolist())
print("Test feature rows:", len(test_features))

assert len(test_features) == 8

## 16. Fit final supervised models on ALL standard training cases

The models now train on Cases 01, 02, 03, 05 and 06.

Case 06 is explicitly included.

In [ ]:
X_train = feature_df[MODEL_FEATURES].copy()

training_medians = X_train.median(numeric_only=True)

X_train_clean = (
    X_train
    .fillna(training_medians)
    .fillna(0)
)

y_train = feature_df["faulty"]

fitted_models = {}

for name, base_model in MODELS.items():
    model = clone(base_model)
    model.fit(X_train_clean, y_train)
    fitted_models[name] = model

print("Final models fitted on:", sorted(feature_df["filename"].unique()))
assert "acv_case_06.xlsx" in feature_df["filename"].values

## 17. Frozen final ensemble inference

In [ ]:
def final_inference(
    raw_df,
    case_features,
    fitted_models,
    training_medians,
    method_weights=LOCKED_METHOD_WEIGHTS,
):
    result = case_features[["car"]].copy()

    # Domain
    domain = add_domain_score(case_features)

    result["domain"] = normalized_rank_score(
        domain["domain_score"]
    ).values

    # Supervised
    X = (
        case_features[MODEL_FEATURES]
        .fillna(training_medians)
        .fillna(0)
    )

    for name, model in fitted_models.items():
        probability = model.predict_proba(X)[:, 1]

        result[name] = normalized_rank_score(
            probability
        ).values

    # Persistence
    for minutes, name in [
        (30, "window30"),
        (60, "window60"),
    ]:
        window_scores = window_aggregate(
            raw_df,
            minutes=minutes,
        )

        merged = result[["car"]].merge(
            window_scores[["car", "window_score"]],
            on="car",
            how="left",
        )

        result[name] = normalized_rank_score(
            merged["window_score"].fillna(0.5)
        ).values

    result["ensemble_score"] = sum(
        method_weights[name] * result[name]
        for name in method_weights
    )

    result = (
        result
        .sort_values("ensemble_score", ascending=False)
        .reset_index(drop=True)
    )

    result["predicted_rank"] = np.arange(
        1,
        len(result) + 1,
    )

    # Simple agreement diagnostic:
    method_cols = list(method_weights.keys())

    result["methods_ranking_car_first"] = 0

    for method in method_cols:
        top_car = result.loc[
            result[method].idxmax(),
            "car",
        ]

        result.loc[
            result["car"].eq(top_car),
            "methods_ranking_car_first",
        ] += 1

    return result


final_ranking = final_inference(
    raw_df=test_df,
    case_features=test_features,
    fitted_models=fitted_models,
    training_medians=training_medians,
)

final_ranking

## 18. Confidence diagnostics

There is no published hidden test label, so this is **not** an accuracy probability.

We report:

- the ensemble-score margin between rank #1 and rank #2,
- how many of the six component methods independently rank the final top car first.

In [ ]:
top1 = final_ranking.iloc[0]
top2 = final_ranking.iloc[1]

score_margin = (
    top1["ensemble_score"]
    - top2["ensemble_score"]
)

top_car = top1["car"]

method_cols = list(LOCKED_METHOD_WEIGHTS.keys())

method_top_choices = {
    method: final_ranking.loc[
        final_ranking[method].idxmax(),
        "car",
    ]
    for method in method_cols
}

agreement_count = sum(
    car == top_car
    for car in method_top_choices.values()
)

print("Top-ranked car:", top_car)
print("Top-1 vs Top-2 ensemble margin:", round(score_margin, 4))
print(
    "Methods independently ranking the same car #1:",
    f"{agreement_count}/{len(method_cols)}",
)
print("\nMethod top choices:")
print(method_top_choices)

## 19. Generate the required `acv_predictions.csv`

In [ ]:
ranked_cars = final_ranking["car"].tolist()

prediction = pd.DataFrame([{
    "file_id": TEST_FILE.name.replace("(1)", ""),
    "ranked_cars": "|".join(ranked_cars),
}])

# Submission-format checks
assert len(ranked_cars) == 8
assert len(set(ranked_cars)) == 8
assert all(re.fullmatch(r"\d{2}", car) for car in ranked_cars)

prediction_path = OUTPUT_DIR / "acv_predictions.csv"

prediction.to_csv(
    prediction_path,
    index=False,
)

print("Saved:", prediction_path)
display(prediction)

## 20. Save the frozen model bundle for app integration

The app can load this bundle without retraining the supervised models.

In [ ]:
bundle = {
    "models": fitted_models,
    "training_medians": training_medians,
    "model_features": MODEL_FEATURES,
    "robust_features": ROBUST_FEATURES,
    "ensemble_weights": LOCKED_METHOD_WEIGHTS,
    "standard_training_cases": STANDARD_CASES,
    "random_state": RANDOM_STATE,
}

bundle_path = OUTPUT_DIR / "acv_model_bundle.joblib"

joblib.dump(
    bundle,
    bundle_path,
)

print("Saved:", bundle_path)

## 21. Save validation evidence

In [ ]:
validation.to_csv(
    OUTPUT_DIR / "acv_locked_validation.csv",
    index=False,
)

ablation_results.to_csv(
    OUTPUT_DIR / "acv_feature_ablation.csv",
    index=False,
)

final_ranking.to_csv(
    OUTPUT_DIR / "acv_test_ranking_diagnostics.csv",
    index=False,
)

if stress_summary is not None:
    stress_summary.to_csv(
        OUTPUT_DIR / "acv_stress_summary.csv"
    )

print("Validation and diagnostic files saved to:", OUTPUT_DIR)

## 22. Final ranking visual

In [ ]:
plot_df = final_ranking.sort_values(
    "ensemble_score",
    ascending=True,
)

plt.figure(figsize=(9, 5))
plt.barh(
    plot_df["car"],
    plot_df["ensemble_score"],
)

plt.xlabel("Locked Ensemble Suspicion Score")
plt.ylabel("Car")
plt.title("ACV Refrigerant-Leak Ranking")
plt.show()

# Final methodology

This version is designed specifically to reduce overfitting risk.

**Training:** Cases 01, 02, 03, 05 and 06 only for the standard-schema pipeline.

**Case 04:** excluded from the main model because its rich telemetry schema is materially different and a forced mapping could introduce incorrect assumptions.

**Features:** fixed thermal, peer-relative and persistence signals.

**Validation:** leave one entire fault case out at a time.

**Supervised models:** conservative Logistic Regression, Random Forest and Extra Trees.

**Unsupervised/domain components:** peer-relative thermal score plus 30- and 60-minute persistence analysis.

**Ensemble:** equal-weight rank consensus, frozen before the test file is read.

**Test use:** inference only. The hidden test label is not available and is never used.

This means a perfect local validation score should still be interpreted cautiously because there are only five directly comparable independent fault cases. The main purpose of this pipeline is to make the prediction stable, explainable and leakage-resistant rather than to optimise aggressively on a tiny sample.